# 01 — Dual Cell Geometry Visualization

Visualize primal mesh + dual mesh, highlight individual dual cell polygons,
and compare the two polygon formulations:
- **barycentric_dual_p_ij**: barycenters + edge midpoints (standard DEC)
- **barycentric**: barycenters only

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

from hyperct import Complex
from hyperct.ddg import compute_vd, d_area
from hyperct.ddg.plot_dual import plot_dual_mesh_1D, plot_dual_mesh_2D
from hyperct.ddg._dual_cell import (
    dual_cell_vertices_1d, dual_cell_polygon_2d,
    dual_cell_area_2d, _shoelace_area,
)

## 1D Dual Cells

In [ ]:
HC = Complex(1, domain=[(0.0, 1.0)])
HC.triangulate()
HC.refine_all()
HC.refine_all()

bV = set()
for v in HC.V:
    on_bnd = abs(v.x_a[0]) < 1e-14 or abs(v.x_a[0] - 1.0) < 1e-14
    v.boundary = on_bnd
    if on_bnd: bV.add(v)

compute_vd(HC, cdist=1e-10)

fig, ax = plt.subplots(figsize=(10, 2))
plot_dual_mesh_1D(HC, ax=ax, show=False)

# Highlight dual cells of interior vertices
colors = plt.cm.Set2(np.linspace(0, 1, 8))
for i, v in enumerate(v for v in HC.V if v not in bV):
    a, b = dual_cell_vertices_1d(v)
    ax.axvspan(a, b, alpha=0.2, color=colors[i % 8])
    ax.plot(v.x_a[0], 0, 'ko', markersize=8, zorder=5)
    ax.annotate(f'[{a:.3f}, {b:.3f}]', (v.x_a[0], 0.15),
               ha='center', fontsize=8)

ax.set_title('1D Dual Cells (shaded regions)')
ax.set_ylim(-0.3, 0.5)
plt.tight_layout()
plt.show()

## 2D Dual Cells — Primal + Dual Mesh Overlay

In [ ]:
HC2 = Complex(2, domain=[(0.0, 1.0), (0.0, 1.0)])
HC2.triangulate()
HC2.refine_all()

bV2 = set()
for v in HC2.V:
    on_bnd = any(abs(v.x_a[d] - lo) < 1e-14 or abs(v.x_a[d] - hi) < 1e-14
                 for d, (lo, hi) in enumerate([(0,1),(0,1)]))
    v.boundary = on_bnd
    if on_bnd: bV2.add(v)

compute_vd(HC2, method='barycentric', cdist=1e-10)

fig, ax = plt.subplots(figsize=(8, 8))
plot_dual_mesh_2D(HC2, ax=ax, show=False)
ax.set_title('2D Primal (black) + Dual (blue) Mesh')
plt.tight_layout()
plt.show()

## 2D — Highlight Individual Dual Cell Polygons

In [ ]:
interior = [v for v in HC2.V if v not in bV2]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, include_mp, title in zip(
    axes,
    [True, False],
    ['barycentric_dual_p_ij (with edge midpoints)',
     'barycentric (duals only)']
):
    plot_dual_mesh_2D(HC2, ax=ax, show=False)

    patches = []
    colors_list = []
    for i, v in enumerate(interior):
        polygon = dual_cell_polygon_2d(v, include_edge_midpoints=include_mp)
        area = abs(_shoelace_area(polygon))
        patch = MplPolygon(polygon, closed=True)
        patches.append(patch)
        colors_list.append(area)

        # Mark polygon vertices
        ax.plot(polygon[:, 0], polygon[:, 1], 'g.', markersize=4)
        # Mark primal vertex
        ax.plot(v.x_a[0], v.x_a[1], 'ro', markersize=6, zorder=5)
        # Label area
        ax.annotate(f'{area:.4f}', v.x_a[:2], fontsize=7,
                    ha='center', va='bottom', color='red')

    pc = PatchCollection(patches, alpha=0.3, cmap='coolwarm')
    pc.set_array(np.array(colors_list))
    ax.add_collection(pc)
    ax.set_title(title)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## Area Comparison: d_area (approx) vs shoelace (exact)

In [ ]:
print(f'{"Vertex":>16}  {"d_area (approx)":>15}  {"shoelace (p_ij)":>15}  {"shoelace (bary)":>15}  {"ratio p_ij/d":>12}')
print('-' * 82)
for v in interior:
    da = d_area(v)
    sa_pij = dual_cell_area_2d(v, include_edge_midpoints=True)
    sa_bary = dual_cell_area_2d(v, include_edge_midpoints=False)
    ratio = sa_pij / da if da > 0 else float('nan')
    print(f'{str(v.x):>16}  {da:>15.8f}  {sa_pij:>15.8f}  {sa_bary:>15.8f}  {ratio:>12.4f}')

print(f'\nSum d_area:        {sum(d_area(v) for v in interior):.8f}')
print(f'Sum shoelace p_ij: {sum(dual_cell_area_2d(v, True) for v in interior):.8f}')
print(f'Sum shoelace bary: {sum(dual_cell_area_2d(v, False) for v in interior):.8f}')
print(f'Domain area:       1.00000000')

## Circumcentric Dual — Same Visualization

In [ ]:
HC_c = Complex(2, domain=[(0.0, 1.0), (0.0, 1.0)])
HC_c.triangulate()
HC_c.refine_all()

bV_c = set()
for v in HC_c.V:
    on_bnd = any(abs(v.x_a[d] - lo) < 1e-14 or abs(v.x_a[d] - hi) < 1e-14
                 for d, (lo, hi) in enumerate([(0,1),(0,1)]))
    v.boundary = on_bnd
    if on_bnd: bV_c.add(v)

compute_vd(HC_c, method='circumcentric', cdist=1e-10)
interior_c = [v for v in HC_c.V if v not in bV_c]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, HC_plot, int_verts, title in zip(
    axes, [HC2, HC_c], [interior, interior_c],
    ['Barycentric dual', 'Circumcentric dual']
):
    plot_dual_mesh_2D(HC_plot, ax=ax, show=False)
    for v in int_verts:
        polygon = dual_cell_polygon_2d(v, include_edge_midpoints=True)
        patch = MplPolygon(polygon, closed=True, alpha=0.2,
                           facecolor='C0', edgecolor='C0')
        ax.add_patch(patch)
        ax.plot(v.x_a[0], v.x_a[1], 'ro', markersize=5)
    ax.set_title(title)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()